In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.multivariate.manova import MANOVA

DATA_FILE = "/Users/shawnlam/Everything GRAD/Pilot_Adults_BWforpython.xlsx"
alpha = 0.05

# sheet_name="Main" reads only that sheet; the other sheets are ignored.
main = pd.read_excel(DATA_FILE, sheet_name="Main")

In [ ]:
main["BW_change_pct"] = (main["Day_40"] - main["Day_1"]) / main["Day_1"] * 100

# Saline and MC pooled as "Control", as in your earlier thyroid analysis.
main["Group"] = main["Drug"].replace({"Saline": "Control", "MC": "Control"})
order = ["Control", "MMI-L", "MMI-H", "PTU-L", "PTU-H"]
main["Group"] = pd.Categorical(main["Group"], categories=order, ordered=True)

outcomes = ["Average", "BW_change_pct"]
main.groupby("Group", observed=True)[outcomes].agg(["count", "mean", "std"]).round(3)

In [ ]:
rows = []
for var in outcomes:
    samples = [g[var].to_numpy() for _, g in main.groupby("Group", observed=True)]
    rows.append({
        "outcome": var,
        "Brown-Forsythe p": stats.levene(*samples, center="median").pvalue,  # robust to non-normal data
        "Levene p": stats.levene(*samples, center="mean").pvalue,            # classic version
        "Bartlett p": stats.bartlett(*samples).pvalue,                       # sensitive to non-normality
    })
variance_tests = pd.DataFrame(rows).set_index("outcome").round(3)
variance_tests["violated (any p < alpha)"] = (variance_tests < alpha).any(axis=1)
variance_tests

In [ ]:
def correlation_report(d, label):
    # 1. Overall correlation (ignores group)
    r, p = stats.pearsonr(d["Average"], d["BW_change_pct"])
    rho, p_rho = stats.spearmanr(d["Average"], d["BW_change_pct"])
    # 2. Within-group correlation: remove each group's mean from both variables,
    #    then correlate what is left (the "residuals").
    res_a = smf.ols("Average ~ C(Group)", d).fit().resid
    res_b = smf.ols("BW_change_pct ~ C(Group)", d).fit().resid
    r_w = np.corrcoef(res_a, res_b)[0, 1]
    df_w = len(d) - d["Group"].nunique() - 2                  # degrees of freedom after removing group means
    p_w = 2 * stats.t.sf(abs(r_w * np.sqrt(df_w / (1 - r_w ** 2))), df_w)
    rho_w, p_rho_w = stats.spearmanr(res_a, res_b)
    return {"data": label, "n": len(d),
            "Pearson r": r, "p": p, "Spearman rho": rho, "p (S)": p_rho,
            "within-group r": r_w, "p (W)": p_w,
            "within-group rho": rho_w, "p (WS)": p_rho_w}

# Repeat without the animal with the largest thyroid, to see whether one animal drives the result.
extreme = main.loc[main["Average"].idxmax(), "Subject"]
results = pd.DataFrame([
    correlation_report(main, "all animals"),
    correlation_report(main[main["Subject"] != extreme], f"without {extreme}"),
]).set_index("data").round(3)
results.T

In [ ]:
import numpy as np
from scipy import stats
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.multitest import multipletests

def overall_tests(var):
    # Split the outcome into one array of values per group.
    groups = [main.loc[main["Group"] == g, var].to_numpy() for g in order]

    # Equal-variance check (Brown-Forsythe: robust to non-normal data).
    levene_p = stats.levene(*groups, center="median").pvalue

    # Standard one-way ANOVA (assumes equal variances).
    f_stat, p_anova = stats.f_oneway(*groups)

    # Effect size eta-squared: share of the total variation explained by group.
    all_values = np.concatenate(groups)
    ss_total = ((all_values - all_values.mean()) ** 2).sum()
    ss_between = sum(len(g) * (g.mean() - all_values.mean()) ** 2 for g in groups)

    # Welch's ANOVA (does not assume equal variances).
    welch = anova_oneway(groups, use_var="unequal", welch_correction=True)

    # Kruskal-Wallis (rank-based, no normality assumption).
    h_stat, p_kw = stats.kruskal(*groups)

    return {"outcome": var,
            "Levene p": levene_p,
            "ANOVA F": f_stat, "ANOVA p": p_anova, "eta squared": ss_between / ss_total,
            "Welch F": welch.statistic, "Welch p": welch.pvalue,
            "Kruskal-Wallis H": h_stat, "KW p": p_kw}

overall = pd.DataFrame([overall_tests(v) for v in outcomes]).set_index("outcome")

# Two outcomes were tested, so multiply p-values by 2 (Bonferroni) to keep the
# overall false-positive rate at 5%. Values are capped at 1.
for col in ["ANOVA p", "Welch p", "KW p"]:
    overall[col + " (x2)"] = (overall[col] * 2).clip(upper=1)

overall.round(4).T

In [ ]:
def posthoc_vs_control(var):
    control = main.loc[main["Group"] == "Control", var].to_numpy()
    treated = [g for g in order if g != "Control"]
    samples = [main.loc[main["Group"] == g, var].to_numpy() for g in treated]

    # Dunnett's test: compares each treated group with Control, corrected for 4 comparisons.
    dunnett = stats.dunnett(*samples, control=control)
    ci = dunnett.confidence_interval()

    # Welch t-test vs Control (allows unequal variance), then Holm correction for 4 tests.
    welch_p = [stats.ttest_ind(s, control, equal_var=False).pvalue for s in samples]
    welch_holm = multipletests(welch_p, method="holm")[1]

    # Mann-Whitney vs Control (rank-based), then Holm correction.
    mw_p = [stats.mannwhitneyu(s, control).pvalue for s in samples]
    mw_holm = multipletests(mw_p, method="holm")[1]

    return pd.DataFrame({
        "n": [len(s) for s in samples],
        "diff_vs_Control": [s.mean() - control.mean() for s in samples],
        "CI_low": ci.low, "CI_high": ci.high,
        "Dunnett p": dunnett.pvalue,
        "Welch (Holm) p": welch_holm,
        "Mann-Whitney (Holm) p": mw_holm,
    }, index=treated).round(4)

for v in outcomes:
    print("\n==", v, "(control mean =", round(main.loc[main.Group == "Control", v].mean(), 4), ")")
    display(posthoc_vs_control(v))

In [ ]:
import matplotlib.pyplot as plt

def stars(p):
    """Convert a p-value into the usual star label."""
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

# Same colours as the scatter plot, so each group looks the same in every figure.
colours = {"Control": "#4D4D4D", "MMI-L": "#56B4E9", "MMI-H": "#0072B2",
           "PTU-L": "#E69F00", "PTU-H": "#D55E00"}

def bar_panel(ax, var, ylabel, title):
    """Draw mean +/- SEM bars, individual animals and Dunnett stars for one outcome on one axis."""
    summary = (main.groupby("Group", observed=True)[var]
                   .agg(mean="mean", sem="sem", n="count")
                   .loc[order])
    dunnett_p = posthoc_vs_control(var)["Dunnett p"]       # from Cell B

    x = np.arange(len(summary))
    ax.bar(x, summary["mean"], yerr=summary["sem"], capsize=5, width=0.6,
           color=[colours[g] for g in summary.index], alpha=0.55, edgecolor="black", zorder=2)

    rng = np.random.default_rng(0)                         # same jitter every time you run it
    for i, group in enumerate(summary.index):
        values = main.loc[main["Group"] == group, var]
        ax.scatter(i + rng.uniform(-0.12, 0.12, len(values)), values, color="black", s=20, zorder=3)

    # Room for the star labels: the axis starts at 0 (or below if some values are negative).
    top, bottom = main[var].max(), min(0, main[var].min())
    span = top - bottom
    for i, group in enumerate(summary.index):
        if group != "Control":                             # Control is the reference: no label
            ax.text(i, top + 0.05 * span, stars(dunnett_p[group]), ha="center", va="bottom", fontsize=13)

    ax.axhline(0, color="black", linewidth=0.8, zorder=1)
    ax.set_ylim(bottom - (0.08 * span if bottom < 0 else 0), top + 0.2 * span)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{g}\n(n={n})" for g, n in zip(summary.index, summary["n"])])
    ax.set_ylabel(ylabel)
    ax.set_title(title)

# Two panels side by side, each with its own y-axis scale.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
bar_panel(ax1, "Average", "Thyroid size (mm²)", "Thyroid size")
bar_panel(ax2, "BW_change_pct", "Body weight change, Day 1 to 40 (%)", "Body weight change")

# Panel letters A and B for the figure legend or caption.
for ax, letter in [(ax1, "A"), (ax2, "B")]:
    ax.text(-0.14, 1.04, letter, transform=ax.transAxes, fontsize=16, fontweight="bold")

fig.suptitle("Bars: mean ± SEM; dots: individual animals. Stars: Dunnett vs Control "
             "(* p<0.05, ** p<0.01, *** p<0.001, ns not significant)",
             fontsize=10, y=0.995)
plt.tight_layout()
plt.savefig("thyroid_and_bw_by_group.png", dpi=300)
plt.show()